In [82]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import os
import shutil
import os
print(os.environ["FINN_BUILD_DIR"])
pynq_part_map = dict()
pynq_part_map["Ultra96"] = "xczu3eg-sbva484-1-e"
pynq_part_map["Ultra96-V2"] = "xczu3eg-sbva484-1-i"
pynq_part_map["Pynq-Z1"] = "xc7z020clg400-1"
pynq_part_map["Pynq-Z2"] = "xc7z020clg400-1"
pynq_part_map["ZCU102"] = "xczu9eg-ffvb1156-2-e"
pynq_part_map["ZCU104"] = "xczu7ev-ffvc1156-2-e"
pynq_part_map["ZCU111"] = "xczu28dr-ffvg1517-2-e"
pynq_part_map["RFSoC2x2"] = "xczu28dr-ffvg1517-2-e"
pynq_part_map["RFSoC4x2"] = "xczu48dr-ffvg1517-2-e"
pynq_part_map["KV260_SOM"] = "xck26-sfvc784-2LV-c"
finn_root = os.getcwd()
print(finn_root)

/home/changhong/prj/finn/notebooks/EF-US-Engine-Free-Unstructured-Sparsity-Design-Alleviates-Accelerator-Bottlenecks/casestudy1/LeNet_MNIST_BNN/build/auto2/home/changhong/prj/finn/notebooks/EF-US-Engine-Free-Unstructured-Sparsity-Design-Alleviates-Accelerator-Bottlenecks/casestudy1/LeNet_MNIST_BNN/tmp/auto2
/home/changhong/prj/finn/notebooks


In [83]:
try_name = "/auto2"

notebook_name = "/EF-US-Engine-Free-Unstructured-Sparsity-Design-Alleviates-Accelerator-Bottlenecks/casestudy1/LeNet_MNIST_BNN"
#finn_root = os.getcwd()
finn_root = "/home/changhong/prj/finn/notebooks"

build_dir = finn_root+ notebook_name +"/build" + try_name
model_dir = finn_root+ notebook_name +"/model"
data_dir = finn_root+ notebook_name +"/data"
estimates_output_dir = finn_root + notebook_name + "/estimates_output" + try_name
model_name = "/lenet_mnist_1w1a_prune_ready2.onnx"
rtlsim_output_dir = finn_root + notebook_name + "/rtlsim_output" + try_name
folding_config_file = finn_root + notebook_name + "/folding_config/auto.json"
tmp_path = build_dir + finn_root+ notebook_name + "/tmp" + try_name

# Create directories if they do not exist
os.makedirs(build_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)
os.makedirs(estimates_output_dir, exist_ok=True)
os.makedirs(rtlsim_output_dir, exist_ok=True)

os.environ["FINN_BUILD_DIR"] = tmp_path
print(f"Data directory: {data_dir}")
print(f"Finn root directory: {finn_root}")
print(f"Build directory: {build_dir}")
print(f"Model directory: {model_dir}")
print(f"Estimates output directory: {estimates_output_dir}")
print(f"RTLSim output directory: {rtlsim_output_dir}")
print(f"Tmp directory: {tmp_path}")

Data directory: /home/changhong/prj/finn/notebooks/EF-US-Engine-Free-Unstructured-Sparsity-Design-Alleviates-Accelerator-Bottlenecks/casestudy1/LeNet_MNIST_BNN/data
Finn root directory: /home/changhong/prj/finn/notebooks
Build directory: /home/changhong/prj/finn/notebooks/EF-US-Engine-Free-Unstructured-Sparsity-Design-Alleviates-Accelerator-Bottlenecks/casestudy1/LeNet_MNIST_BNN/build/auto2
Model directory: /home/changhong/prj/finn/notebooks/EF-US-Engine-Free-Unstructured-Sparsity-Design-Alleviates-Accelerator-Bottlenecks/casestudy1/LeNet_MNIST_BNN/model
Estimates output directory: /home/changhong/prj/finn/notebooks/EF-US-Engine-Free-Unstructured-Sparsity-Design-Alleviates-Accelerator-Bottlenecks/casestudy1/LeNet_MNIST_BNN/estimates_output/auto2
RTLSim output directory: /home/changhong/prj/finn/notebooks/EF-US-Engine-Free-Unstructured-Sparsity-Design-Alleviates-Accelerator-Bottlenecks/casestudy1/LeNet_MNIST_BNN/rtlsim_output/auto2
Tmp directory: /home/changhong/prj/finn/notebooks/EF-US

In [84]:
import torch
from torch.nn import BatchNorm1d
from torch.nn import BatchNorm2d
from torch.nn import MaxPool2d, AvgPool2d
from torch.nn import Module
from torch.nn import ModuleList

from brevitas.core.restrict_val import RestrictValueType
from brevitas.nn import QuantConv2d
from brevitas.nn import QuantIdentity
from brevitas.nn import QuantLinear

from brevitas_examples.bnn_pynq.models.common import CommonActQuant
from brevitas_examples.bnn_pynq.models.common import CommonWeightQuant
from brevitas_examples.bnn_pynq.models.tensor_norm import TensorNorm
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from finn.util.basic import make_build_dir
from finn.util.visualization import showInNetron
import os
import configparser
import brevitas.nn as qnn
from brevitas.core.quant import QuantType
from brevitas.core.restrict_val import RestrictValueType
from brevitas.core.scaling import ScalingImplType
# CNV_OUT_CH_POOL = [(64, False), (64, True), (128, False), (128, True), (256, False), (256, False)]
# INTERMEDIATE_FC_FEATURES = [(256, 512), (512, 512)]
# LAST_FC_IN_FEATURES = 512
# LAST_FC_PER_OUT_CH_SCALING = False
# POOL_SIZE = 2
# KERNEL_SIZE = 3

# LeNet-5
CNV_OUT_CH_POOL = [(6, True), (16, True), (120, False)]  
INTERMEDIATE_FC_FEATURES = [(120, 84)]  
LAST_FC_IN_FEATURES = 84 
LAST_FC_PER_OUT_CH_SCALING = False
POOL_SIZE = 2  
KERNEL_SIZE = 5  

model_name = '2c3f1w1a_mnist'

class CNV(Module):

    def __init__(self, num_classes, weight_bit_width, act_bit_width, in_bit_width, in_ch):
        super(CNV, self).__init__()

        self.conv_features = ModuleList()
        self.linear_features = ModuleList()


        # self.qrelu = qnn.QuantReLU(quant_type=QuantType.INT, 
        #                     bit_width=act_bit_width, 
        #                     max_val= 1- 1/128.0,
        #                     restrict_scaling_type=RestrictValueType.POWER_OF_TWO,
        #                     scaling_impl_type=ScalingImplType.CONST )

        # self.qrelu = qnn.QuantReLU(
        #             #quant_type=QuantType.INT, 
        #             bit_width=act_bit_width, 
        #             min_val=- 1.0,
        #             max_val=1.0 - 2.0 ** (-7),
        #             restrict_scaling_type=RestrictValueType.POWER_OF_TWO,
        #             scaling_impl_type=ScalingImplType.CONST )
        self.act = qnn.QuantHardTanh(
            bit_width=act_bit_width,
            max_val=1.0,
            min_val=-1.0,
            restrict_scaling_type=RestrictValueType.POWER_OF_TWO)





        self.conv_features.append(QuantIdentity( # for Q1.7 input format
            act_quant=CommonActQuant,
            bit_width=in_bit_width,
            min_val=- 1.0,
            max_val=1.0 - 2.0 ** (-7),
            narrow_range=False,
            restrict_scaling_type=RestrictValueType.POWER_OF_TWO))

        for out_ch, is_pool_enabled in CNV_OUT_CH_POOL:
            self.conv_features.append(
                QuantConv2d(
                    kernel_size=KERNEL_SIZE,
                    in_channels=in_ch,
                    out_channels=out_ch,
                    bias=False,
                    weight_quant=CommonWeightQuant,
                    weight_bit_width=weight_bit_width))
            in_ch = out_ch
            #
            # self.conv_features.append(BatchNorm2d(in_ch, eps=1e-4))
            # self.conv_features.append(
            #     QuantIdentity(act_quant=CommonActQuant, bit_width=act_bit_width))
            self.conv_features.append(self.act)
            #
            if is_pool_enabled:
                self.conv_features.append(MaxPool2d(kernel_size=2))
                #self.conv_features.append(AvgPool2d(kernel_size=2))

        for in_features, out_features in INTERMEDIATE_FC_FEATURES:
            self.linear_features.append(
                QuantLinear(
                    in_features=in_features,
                    out_features=out_features,
                    bias=False,
                    weight_quant=CommonWeightQuant,
                    weight_bit_width=weight_bit_width))
            #
            # self.linear_features.append(BatchNorm1d(out_features, eps=1e-4))
            # self.linear_features.append(
            #     QuantIdentity(act_quant=CommonActQuant, bit_width=act_bit_width))
            self.linear_features.append(self.act)
            #

        self.linear_features.append(
            QuantLinear(
                in_features=LAST_FC_IN_FEATURES,
                out_features=num_classes,
                bias=False,
                weight_quant=CommonWeightQuant,
                weight_bit_width=weight_bit_width))
        self.linear_features.append(TensorNorm())

        for m in self.modules():
            if isinstance(m, QuantConv2d) or isinstance(m, QuantLinear):
                torch.nn.init.uniform_(m.weight.data, -1, 1)

    def clip_weights(self, min_val, max_val):
        for mod in self.conv_features:
            if isinstance(mod, QuantConv2d):
                mod.weight.data.clamp_(min_val, max_val)
        for mod in self.linear_features:
            if isinstance(mod, QuantLinear):
                mod.weight.data.clamp_(min_val, max_val)

    def forward(self, x):
        x = 2.0 * x - torch.tensor([1.0], device=x.device)
        for mod in self.conv_features:
            x = mod(x)
        x = x.view(x.shape[0], -1)
        for mod in self.linear_features:
            x = mod(x)
        return x 


def cnv(cfg):
    weight_bit_width = cfg.getint('QUANT', 'WEIGHT_BIT_WIDTH')
    act_bit_width = cfg.getint('QUANT', 'ACT_BIT_WIDTH')
    in_bit_width = cfg.getint('QUANT', 'IN_BIT_WIDTH')
    num_classes = cfg.getint('MODEL', 'NUM_CLASSES')
    in_channels = cfg.getint('MODEL', 'IN_CHANNELS')
    net = CNV(
        weight_bit_width=weight_bit_width,
        act_bit_width=act_bit_width,
        in_bit_width=in_bit_width,
        num_classes=num_classes,
        in_ch=in_channels)
    return net

config = configparser.ConfigParser()
config['MODEL'] = {
    'NUM_CLASSES': '10',
    'IN_CHANNELS': '1',
    'DTASET': 'MNIST',
}
config['QUANT'] = {
    'WEIGHT_BIT_WIDTH': '2',
    'ACT_BIT_WIDTH': '2',
    'IN_BIT_WIDTH': '8',
}

model = cnv(config)


In [85]:
import numpy as np

ready_model_filename = model_dir + "/lenet_mnist_1w1a_prune_ready2.onnx"

input_shape = (1, 1, 32, 32)

input_a = np.random.randint(0, 1, size=input_shape).astype(np.float32)
input_a = 2 * input_a - 1
scale = 1.0
input_t = torch.from_numpy(input_a * scale)

#Move to CPU before export
model.cpu()

# Export to ONNX
export_qonnx(
    model, export_path=ready_model_filename, input_t=input_t
)

# clean-up
qonnx_cleanup(ready_model_filename, out_file=ready_model_filename)

print("Model saved to %s" % ready_model_filename)

Model saved to /home/changhong/prj/finn/notebooks/EF-US-Engine-Free-Unstructured-Sparsity-Design-Alleviates-Accelerator-Bottlenecks/casestudy1/LeNet_MNIST_BNN/model/lenet_mnist_1w1a_prune_ready2.onnx


In [86]:
if os.path.exists(estimates_output_dir):
    shutil.rmtree(estimates_output_dir)
    print("Previous run results deleted!")

if os.path.exists(rtlsim_output_dir):
    shutil.rmtree(rtlsim_output_dir)
    print("Previous run results deleted!")

if os.path.exists(tmp_path):
    shutil.rmtree(tmp_path)
    print("Previous run results deleted!")

Previous run results deleted!
Previous run results deleted!


In [87]:
cfg_estimates = build.DataflowBuildConfig(
    output_dir          = estimates_output_dir,
    target_fps          = 1000000,
    synth_clk_period_ns = 10.0,
    # fpga_part           = "xc7z020clg400-1",
    fpga_part           = pynq_part_map["ZCU104"],
    steps               = build_cfg.estimate_only_dataflow_steps,
    folding_config_file = folding_config_file,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ]
)

build.build_dataflow_cfg(ready_model_filename, cfg_estimates)

Building dataflow accelerator from /home/changhong/prj/finn/notebooks/EF-US-Engine-Free-Unstructured-Sparsity-Design-Alleviates-Accelerator-Bottlenecks/casestudy1/LeNet_MNIST_BNN/model/lenet_mnist_1w1a_prune_ready2.onnx
Intermediate outputs will be generated in /home/changhong/prj/finn/notebooks/EF-US-Engine-Free-Unstructured-Sparsity-Design-Alleviates-Accelerator-Bottlenecks/casestudy1/LeNet_MNIST_BNN/build/auto2/home/changhong/prj/finn/notebooks/EF-US-Engine-Free-Unstructured-Sparsity-Design-Alleviates-Accelerator-Bottlenecks/casestudy1/LeNet_MNIST_BNN/tmp/auto2
Final outputs will be generated in /home/changhong/prj/finn/notebooks/EF-US-Engine-Free-Unstructured-Sparsity-Design-Alleviates-Accelerator-Bottlenecks/casestudy1/LeNet_MNIST_BNN/estimates_output/auto2
Build log is at /home/changhong/prj/finn/notebooks/EF-US-Engine-Free-Unstructured-Sparsity-Design-Alleviates-Accelerator-Bottlenecks/casestudy1/LeNet_MNIST_BNN/estimates_output/auto2/build_dataflow.log
Running step: step_qonnx_

0

In [ ]:
cfg_stitched_ip = build.DataflowBuildConfig(
    output_dir          = rtlsim_output_dir,
    mvau_wwidth_max     = 10000,
    target_fps          = 1000000,
    synth_clk_period_ns = 10.0,    
    fpga_part           = pynq_part_map["ZCU104"],
    folding_config_file = folding_config_file,
    generate_outputs=[
        build_cfg.DataflowOutputType.STITCHED_IP,
        build_cfg.DataflowOutputType.RTLSIM_PERFORMANCE,
        build_cfg.DataflowOutputType.OOC_SYNTH,
    ]
)

build.build_dataflow_cfg(ready_model_filename, cfg_stitched_ip)